In [ ]:
!pip install datasets transformers manga109api Pillow torch torchvision ultralytics

from google.colab import userdata
from datasets import load_dataset
import zipfile
import manga109api
import os

from ultralytics import YOLO
from PIL import Image


from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))


from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="hal-utokyo/Manga109",
    repo_type="dataset",
    local_dir="/content/manga109"
)
print(path)

for root, dirs, files in os.walk("/content/manga109"):
    level = root.replace("/content/manga109", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # only show files for top 2 levels
        for f in files[:5]:  # limit to 5 files per folder
            print(f"{indent}  {f}")



zip_path = "/content/manga109/Manga109_released_2023_12_07.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    print(z.namelist()[:20])  # preview contents first


extract_path = "/content/manga109_data"

with zipfile.ZipFile(zip_path, 'r') as z:
    members = [m for m in z.namelist() if not m.startswith("__MACOSX")]
    z.extractall(extract_path, members=members)

print("Done!")

data_root = "/content/manga109_data/Manga109_released_2023_12_07"
api = manga109api.Parser(root_dir=data_root)

# List all books
print(api.books)


import os
from PIL import Image

LABEL2ID = {"body": 0, "text": 1}

def convert_all_annotations(api, data_root, label_out_root):
    image_dir = os.path.join(data_root, "images")

    for book in api.books:
        ann = api.get_annotation(book=book)

        for page in ann['page']:
            page_index = page['@index']
            W = page['@width']
            H = page['@height']

            img_filename = f"{page_index:03d}.jpg"
            img_path = os.path.join(image_dir, book, img_filename)
            if not os.path.exists(img_path):
                continue

            lines = []
            for tag, label_id in LABEL2ID.items():
                for item in page[tag]:
                    x1 = item['@xmin']
                    y1 = item['@ymin']
                    x2 = item['@xmax']
                    y2 = item['@ymax']

                    cx = ((x1 + x2) / 2) / W
                    cy = ((y1 + y2) / 2) / H
                    w  = (x2 - x1) / W
                    h  = (y2 - y1) / H
                    lines.append(f"{label_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

            label_out = os.path.join(label_out_root, book)
            os.makedirs(label_out, exist_ok=True)
            with open(os.path.join(label_out, img_filename.replace(".jpg", ".txt")), "w") as f:
                f.write("\n".join(lines))

    print("Annotation conversion done!")

label_out_root = "/content/manga109_labels"
convert_all_annotations(api, data_root, label_out_root)


# Verify a label file looks right
import random
book = random.choice(api.books)
label_file = f"/content/manga109_labels/{book}/002.txt"
if os.path.exists(label_file):
    with open(label_file) as f:
        print(f.read())


import shutil, yaml, os

data_root = "/content/manga109_data/Manga109_released_2023_12_07"
image_dir = os.path.join(data_root, "images")
label_out_root = "/content/manga109_labels"
dataset_root = "/content/manga109_dataset"

# Train/val split (80/20)
all_books = api.books
split = int(len(all_books) * 0.8)
train_books = all_books[:split]
val_books = all_books[split:]

print(f"Train: {len(train_books)} books, Val: {len(val_books)} books")

# Copy images and labels into dataset folder
for split_name, books in [("train", train_books), ("val", val_books)]:
    img_split_dir = os.path.join(dataset_root, "images", split_name)
    lbl_split_dir = os.path.join(dataset_root, "labels", split_name)
    os.makedirs(img_split_dir, exist_ok=True)
    os.makedirs(lbl_split_dir, exist_ok=True)

    for book in books:
        src_img = os.path.join(image_dir, book)
        src_lbl = os.path.join(label_out_root, book)
        dst_img = os.path.join(img_split_dir, book)
        dst_lbl = os.path.join(lbl_split_dir, book)

        if os.path.exists(src_img) and not os.path.exists(dst_img):
            shutil.copytree(src_img, dst_img)
        if os.path.exists(src_lbl) and not os.path.exists(dst_lbl):
            shutil.copytree(src_lbl, dst_lbl)

print("Dataset split done!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

/content/manga109
manga109/
  .gitattributes
  README.md
  Manga109_released_2023_12_07.zip
  .cache/
    huggingface/
      download/
['Manga109_released_2023_12_07/', '__MACOSX/._Manga109_released_2023_12_07', 'Manga109_released_2023_12_07/books.txt', '__MACOSX/Manga109_released_2023_12_07/._books.txt', 'Manga109_released_2023_12_07/annotations.v2020.12.18/', '__MACOSX/Manga109_released_2023_12_07/._annotations.v2020.12.18', 'Manga109_released_2023_12_07/images/', '__MACOSX/Manga109_released_2023_12_07/._images', 'Manga109_released_2023_12_07/annotations.v2018.05.31/', '__MACOSX/Manga109_released_2023_12_07/._annotations.v2018.05.31', 'Manga109_released_2023_12_07/annotations/', '__MACOSX/Manga109_released_2023_12_07/._annotations', 'Manga109_released_2023_12_07/readme.txt', '__MACOSX/Manga109_released_2023_12_07/._readme.txt', 'Manga109_released_2023_12_07/annotations_COO/', '__MACOSX/Manga109_released_2023_12_07/._annotations_COO', 'Manga109_released_2023_12_07/annotations_Manga109

In [ ]:
#ai was used to heavily fix my base code with XGBoosting.

# -*- coding: utf-8 -*-
"""
YOLOv8s + XGBoost Post-Processing Pipeline
===========================================
Based on Ahmad & Schich (2023) "Toward cross-domain object detection
in artwork images using improved YoloV5 and XGBoosting"

Pipeline:
  Phase 1: Train YOLOv8s normally (or use existing trained weights)
  Phase 2: Run YOLO on training set, extract per-detection features,
           label each detection as TP or FP based on IoU with GT
  Phase 3: Train XGBoost classifier on these features
  Phase 4: At inference, YOLO proposes → XGBoost filters → final output

This suppresses false positives and emphasizes hard-to-detect samples.
"""

# ══════════════════════════════════════════════════════════════════
#  STEP 0: INSTALLS AND IMPORTS
# ══════════════════════════════════════════════════════════════════

# !pip install xgboost  # run this if not already installed

import os, shutil, yaml, json, time, glob
import numpy as np
import torch
from ultralytics import YOLO
from collections import defaultdict
import manga109api

try:
    import xgboost as xgb
    print(f"XGBoost {xgb.__version__} ✓")
except ImportError:
    raise ImportError("Run: !pip install xgboost")

from sklearn.metrics import classification_report, roc_auc_score


# ── Paths ──────────────────────────────────────────────────────────
data_root    = "/content/manga109_data/Manga109_released_2023_12_07"
dataset_root = "/content/manga109_dataset"
api          = manga109api.Parser(root_dir=data_root)

# ── Dataset YAML ───────────────────────────────────────────────────
dataset_cfg = {
    "path":  dataset_root,
    "train": "images/train",
    "val":   "images/val",
    "nc":    2,
    "names": {0: "body", 1: "text"},
}
yaml_path = os.path.join(dataset_root, "manga109.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(dataset_cfg, f)
print(f"Dataset YAML written → {yaml_path} ✓")


XGBoost 3.2.0 ✓
Dataset YAML written → /content/manga109_dataset/manga109.yaml ✓


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  STEP 1: LOAD TRAINED YOLO MODEL
#  Use your best existing model (baseline or CBAM)
# ══════════════════════════════════════════════════════════════════

# Point this to your best trained weights
# Options:
#   "/content/runs/manga109_v4_cbam_neck/weights/best.pt"  (CBAM)
#   "/drive/MyDrive/manga109_v4_cbam_neck/weights/best.pt" (from Drive)
#   "yolov8s.pt" then train fresh (uncomment training block below)

#YOLO_WEIGHTS = "/content/runs/manga109_v4_cbam_neck/weights/best.pt"

# If loading from Drive:
from google.colab import drive
drive.mount('/drive')
YOLO_WEIGHTS = "/drive/MyDrive/CV-Comic-Project/baselines/manga109_yolov8/weights/best.pt"

model = YOLO(YOLO_WEIGHTS)
print(f"YOLO loaded from {YOLO_WEIGHTS} ✓")


Mounted at /drive
YOLO loaded from /drive/MyDrive/CV-Comic-Project/baselines/manga109_yolov8/weights/best.pt ✓


In [ ]:




# ══════════════════════════════════════════════════════════════════
#  STEP 2: HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════

def compute_iou(box1, box2):
    """
    Compute IoU between two boxes in xyxy format.
    box1, box2: [x1, y1, x2, y2]
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter

    return inter / (union + 1e-8)


def load_gt_labels(label_path, img_w, img_h):
    """
    Load YOLO format ground truth labels and convert to xyxy.
    Returns list of (class_id, x1, y1, x2, y2).
    """
    boxes = []
    if not os.path.exists(label_path):
        return boxes

    with open(label_path) as f:
        for line in f.readlines():
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls = int(parts[0])
            cx, cy, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            # Convert normalized xywh to pixel xyxy
            x1 = (cx - w / 2) * img_w
            y1 = (cy - h / 2) * img_h
            x2 = (cx + w / 2) * img_w
            y2 = (cy + h / 2) * img_h
            boxes.append((cls, x1, y1, x2, y2))
    return boxes


def extract_detection_features(box, cls_id, conf, img_w, img_h):
    """
    Extract a feature vector for one detection.
    These features help XGBoost learn what 'good' vs 'bad' detections look like.

    Features:
      0: confidence score
      1: box width (normalized)
      2: box height (normalized)
      3: box area (normalized)
      4: aspect ratio (w/h)
      5: box center x (normalized)
      6: box center y (normalized)
      7: class id
      8: distance from image center
      9: box diagonal (normalized)
    """
    x1, y1, x2, y2 = box
    bw = (x2 - x1) / img_w
    bh = (y2 - y1) / img_h
    area = bw * bh
    aspect = bw / (bh + 1e-8)
    cx = ((x1 + x2) / 2) / img_w
    cy = ((y1 + y2) / 2) / img_h
    dist_center = np.sqrt((cx - 0.5) ** 2 + (cy - 0.5) ** 2)
    diagonal = np.sqrt(bw ** 2 + bh ** 2)

    return [
        conf,           # 0: confidence
        bw,             # 1: width
        bh,             # 2: height
        area,           # 3: area
        aspect,         # 4: aspect ratio
        cx,             # 5: center x
        cy,             # 6: center y
        float(cls_id),  # 7: class
        dist_center,    # 8: distance from center
        diagonal,       # 9: diagonal
    ]


# ══════════════════════════════════════════════════════════════════
#  STEP 3: GENERATE TRAINING DATA FOR XGBOOST
#  Run YOLO on training images, match predictions to GT,
#  label each detection as TP (1) or FP (0)
# ══════════════════════════════════════════════════════════════════

def generate_xgb_dataset(model, split="train", iou_threshold=0.5, conf_threshold=0.1):
    """
    Run YOLO on all images in a split, extract per-detection features,
    and label each detection as TP or FP.

    Args:
        model: YOLO model
        split: "train" or "val"
        iou_threshold: IoU threshold to consider a detection as TP
        conf_threshold: minimum confidence to keep a detection

    Returns:
        X: np.array of shape (N, 10) — feature vectors
        y: np.array of shape (N,) — labels (1=TP, 0=FP)
    """
    img_dir = os.path.join(dataset_root, "images", split)
    lbl_dir = os.path.join(dataset_root, "labels", split)

    all_features = []
    all_labels   = []

    # Collect all image paths
    img_paths = []
    for book in sorted(os.listdir(img_dir)):
        book_path = os.path.join(img_dir, book)
        if not os.path.isdir(book_path):
            continue
        for img_file in sorted(os.listdir(book_path)):
            if img_file.endswith(('.jpg', '.png', '.jpeg')):
                img_paths.append((book, img_file))

    print(f"Processing {len(img_paths)} images from {split} split...")

    # Process in batches for speed
    batch_size = 32
    for batch_start in range(0, len(img_paths), batch_size):
        batch = img_paths[batch_start:batch_start + batch_size]
        batch_img_paths = [
            os.path.join(img_dir, book, img_file)
            for book, img_file in batch
        ]

        # Run YOLO inference
        results = model.predict(
            batch_img_paths,
            conf=conf_threshold,
            iou=0.7,
            imgsz=640,
            verbose=False,
            device=0,
        )

        for (book, img_file), result in zip(batch, results):
            img_h, img_w = result.orig_img.shape[:2]

            # Load ground truth
            lbl_file = img_file.replace('.jpg', '.txt').replace('.png', '.txt')
            lbl_path = os.path.join(lbl_dir, book, lbl_file)
            gt_boxes = load_gt_labels(lbl_path, img_w, img_h)
            gt_matched = [False] * len(gt_boxes)

            # Process each detection
            if result.boxes is None or len(result.boxes) == 0:
                continue

            pred_boxes = result.boxes.xyxy.cpu().numpy()
            pred_confs = result.boxes.conf.cpu().numpy()
            pred_cls   = result.boxes.cls.cpu().numpy().astype(int)

            for j in range(len(pred_boxes)):
                box  = pred_boxes[j]
                conf = float(pred_confs[j])
                cls  = int(pred_cls[j])

                # Extract features
                feat = extract_detection_features(box, cls, conf, img_w, img_h)

                # Match to GT: find best IoU with same class
                best_iou = 0
                best_gt  = -1
                for k, (gt_cls, *gt_box) in enumerate(gt_boxes):
                    if gt_cls != cls or gt_matched[k]:
                        continue
                    iou = compute_iou(box, gt_box)
                    if iou > best_iou:
                        best_iou = iou
                        best_gt  = k

                # Label: TP if IoU > threshold, FP otherwise
                if best_iou >= iou_threshold and best_gt >= 0:
                    label = 1  # True Positive
                    gt_matched[best_gt] = True
                else:
                    label = 0  # False Positive

                all_features.append(feat)
                all_labels.append(label)

        # Progress
        done = min(batch_start + batch_size, len(img_paths))
        if done % (batch_size * 10) == 0 or done >= len(img_paths):
            print(f"  Processed {done}/{len(img_paths)} images")

    X = np.array(all_features)
    y = np.array(all_labels)
    print(f"\n{split} set: {len(X)} detections | TP: {y.sum()} | FP: {(1-y).sum()}")
    return X, y


print("\n── Generating XGBoost training data from YOLO predictions ──")
X_train, y_train = generate_xgb_dataset(model, split="train", conf_threshold=0.1)
X_val,   y_val   = generate_xgb_dataset(model, split="val",   conf_threshold=0.1)






── Generating XGBoost training data from YOLO predictions ──
Processing 8525 images from train split...
  Processed 320/8525 images
  Processed 640/8525 images
  Processed 960/8525 images
  Processed 1280/8525 images
  Processed 1600/8525 images
  Processed 1920/8525 images
  Processed 2240/8525 images
  Processed 2560/8525 images
  Processed 2880/8525 images
  Processed 3200/8525 images
  Processed 3520/8525 images
  Processed 3840/8525 images
  Processed 4160/8525 images
  Processed 4480/8525 images
  Processed 4800/8525 images
  Processed 5120/8525 images
  Processed 5440/8525 images
  Processed 5760/8525 images
  Processed 6080/8525 images
  Processed 6400/8525 images
  Processed 6720/8525 images
  Processed 7040/8525 images
  Processed 7360/8525 images
  Processed 7680/8525 images
  Processed 8000/8525 images
  Processed 8320/8525 images
  Processed 8525/8525 images

train set: 321781 detections | TP: 231522 | FP: 90259
Processing 2077 images from val split...
  Processed 320/207

In [ ]:

# ══════════════════════════════════════════════════════════════════
#  STEP 4: TRAIN XGBOOST CLASSIFIER
# ══════════════════════════════════════════════════════════════════

print("\n── Training XGBoost ──")

# Handle class imbalance (usually many more TPs than FPs from a trained model)
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / (n_pos + 1e-8)
print(f"Class balance: TP={int(n_pos)}, FP={int(n_neg)}, scale_pos_weight={scale_pos_weight:.2f}")

feature_names = [
    'confidence', 'box_width', 'box_height', 'box_area',
    'aspect_ratio', 'center_x', 'center_y', 'class_id',
    'dist_from_center', 'box_diagonal'
]

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    tree_method='hist', # Changed from 'gpu_hist' to 'hist'
    random_state=42,
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=20,
)

# ── Evaluate XGBoost ──────────────────────────────────────────────
y_pred     = xgb_model.predict(X_val)
y_pred_prob = xgb_model.predict_proba(X_val)[:, 1]

print("\n── XGBoost Classification Report ──")
print(classification_report(y_val, y_pred, target_names=['FP', 'TP']))
print(f"AUC-ROC: {roc_auc_score(y_val, y_pred_prob):.4f}")

# ── Feature importance ────────────────────────────────────────────
print("\n── Feature Importance ──")
importances = xgb_model.feature_importances_
for name, imp in sorted(zip(feature_names, importances), key=lambda x: -x[1]):
    print(f"  {name:20s} : {imp:.4f}")




── Training XGBoost ──
Class balance: TP=231522, FP=90259, scale_pos_weight=0.39
[0]	validation_0-logloss:0.63403


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [02:29:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[20]	validation_0-logloss:0.33928
[40]	validation_0-logloss:0.32751
[60]	validation_0-logloss:0.32764
[80]	validation_0-logloss:0.32758
[100]	validation_0-logloss:0.32750
[120]	validation_0-logloss:0.32738
[140]	validation_0-logloss:0.32729
[160]	validation_0-logloss:0.32740
[180]	validation_0-logloss:0.32742
[199]	validation_0-logloss:0.32736

── XGBoost Classification Report ──
              precision    recall  f1-score   support

          FP       0.72      0.88      0.79     21245
          TP       0.95      0.87      0.90     54824

    accuracy                           0.87     76069
   macro avg       0.83      0.87      0.85     76069
weighted avg       0.88      0.87      0.87     76069

AUC-ROC: 0.9328

── Feature Importance ──
  confidence           : 0.8885
  class_id             : 0.0281
  box_width            : 0.0146
  box_area             : 0.0125
  box_diagonal         : 0.0104
  box_height           : 0.0102
  aspect_ratio         : 0.0100
  center_x             :

In [ ]:

# ══════════════════════════════════════════════════════════════════
#  STEP 5: XGBOOST-FILTERED INFERENCE
# ══════════════════════════════════════════════════════════════════

def predict_with_xgboost(model, xgb_model, img_path, conf_threshold=0.1, xgb_threshold=0.5):
    """
    Run YOLO + XGBoost pipeline:
      1. YOLO proposes detections (low conf threshold to get more candidates)
      2. Extract features from each detection
      3. XGBoost scores each detection
      4. Keep only detections where XGBoost says TP

    Args:
        model: YOLO model
        xgb_model: trained XGBoost classifier
        img_path: path to image
        conf_threshold: YOLO confidence threshold (low to get more candidates)
        xgb_threshold: XGBoost probability threshold to keep a detection

    Returns:
        filtered_boxes: list of (x1, y1, x2, y2, conf, cls, xgb_score)
    """
    results = model.predict(img_path, conf=conf_threshold, iou=0.7, imgsz=640, verbose=False)
    result  = results[0]

    if result.boxes is None or len(result.boxes) == 0:
        return []

    img_h, img_w = result.orig_img.shape[:2]
    pred_boxes = result.boxes.xyxy.cpu().numpy()
    pred_confs = result.boxes.conf.cpu().numpy()
    pred_cls   = result.boxes.cls.cpu().numpy().astype(int)

    # Extract features for all detections
    features = []
    for j in range(len(pred_boxes)):
        feat = extract_detection_features(
            pred_boxes[j], pred_cls[j], float(pred_confs[j]), img_w, img_h
        )
        features.append(feat)

    X = np.array(features)

    # XGBoost scoring
    xgb_scores = xgb_model.predict_proba(X)[:, 1]

    # Filter: keep only detections XGBoost thinks are TP
    filtered = []
    for j in range(len(pred_boxes)):
        if xgb_scores[j] >= xgb_threshold:
            filtered.append({
                'box':       pred_boxes[j].tolist(),
                'conf':      float(pred_confs[j]),
                'cls':       int(pred_cls[j]),
                'xgb_score': float(xgb_scores[j]),
            })

    return filtered


In [ ]:


# ══════════════════════════════════════════════════════════════════
#  STEP 6: EVALUATE YOLO+XGBOOST vs YOLO-ONLY
# ══════════════════════════════════════════════════════════════════

def evaluate_pipeline(model, xgb_model, split="val", iou_threshold=0.5,
                      yolo_conf=0.25, xgb_threshold=0.5):
    """
    Compare YOLO-only vs YOLO+XGBoost on the val set.
    Computes precision, recall, and F1 for both.
    """
    img_dir = os.path.join(dataset_root, "images", split)
    lbl_dir = os.path.join(dataset_root, "labels", split)

    # Counters for YOLO-only
    yolo_tp, yolo_fp, yolo_fn = 0, 0, 0
    # Counters for YOLO+XGBoost
    xgb_tp, xgb_fp, xgb_fn = 0, 0, 0

    img_paths = []
    for book in sorted(os.listdir(img_dir)):
        book_path = os.path.join(img_dir, book)
        if not os.path.isdir(book_path):
            continue
        for img_file in sorted(os.listdir(book_path)):
            if img_file.endswith(('.jpg', '.png', '.jpeg')):
                img_paths.append((book, img_file))

    print(f"\nEvaluating on {len(img_paths)} {split} images...")

    batch_size = 32
    for batch_start in range(0, len(img_paths), batch_size):
        batch = img_paths[batch_start:batch_start + batch_size]
        batch_img_paths = [
            os.path.join(img_dir, book, img_file)
            for book, img_file in batch
        ]

        # Run YOLO with LOW conf (XGBoost will filter)
        results_low = model.predict(
            batch_img_paths, conf=0.1, iou=0.7, imgsz=640, verbose=False, device=0
        )
        # Run YOLO with NORMAL conf (for YOLO-only baseline)
        results_normal = model.predict(
            batch_img_paths, conf=yolo_conf, iou=0.7, imgsz=640, verbose=False, device=0
        )

        for (book, img_file), res_low, res_norm in zip(batch, results_low, results_normal):
            img_h, img_w = res_low.orig_img.shape[:2]
            lbl_file = img_file.replace('.jpg', '.txt').replace('.png', '.txt')
            lbl_path = os.path.join(lbl_dir, book, lbl_file)
            gt_boxes = load_gt_labels(lbl_path, img_w, img_h)
            n_gt     = len(gt_boxes)

            # ── YOLO-only evaluation ───────────────────────────────
            if res_norm.boxes is not None and len(res_norm.boxes) > 0:
                norm_boxes = res_norm.boxes.xyxy.cpu().numpy()
                norm_cls   = res_norm.boxes.cls.cpu().numpy().astype(int)
                gt_matched = [False] * n_gt

                for j in range(len(norm_boxes)):
                    best_iou, best_gt = 0, -1
                    for k, (gc, *gb) in enumerate(gt_boxes):
                        if gc != norm_cls[j] or gt_matched[k]:
                            continue
                        iou = compute_iou(norm_boxes[j], gb)
                        if iou > best_iou:
                            best_iou, best_gt = iou, k

                    if best_iou >= iou_threshold and best_gt >= 0:
                        yolo_tp += 1
                        gt_matched[best_gt] = True
                    else:
                        yolo_fp += 1

                yolo_fn += sum(1 for m in gt_matched if not m)
            else:
                yolo_fn += n_gt

            # ── YOLO+XGBoost evaluation ────────────────────────────
            if res_low.boxes is not None and len(res_low.boxes) > 0:
                low_boxes = res_low.boxes.xyxy.cpu().numpy()
                low_confs = res_low.boxes.conf.cpu().numpy()
                low_cls   = res_low.boxes.cls.cpu().numpy().astype(int)

                # Extract features and score with XGBoost
                features = []
                for j in range(len(low_boxes)):
                    feat = extract_detection_features(
                        low_boxes[j], low_cls[j], float(low_confs[j]), img_w, img_h
                    )
                    features.append(feat)

                xgb_scores = xgb_model.predict_proba(np.array(features))[:, 1]

                gt_matched = [False] * n_gt
                for j in range(len(low_boxes)):
                    if xgb_scores[j] < xgb_threshold:
                        continue  # XGBoost filtered this detection out

                    best_iou, best_gt = 0, -1
                    for k, (gc, *gb) in enumerate(gt_boxes):
                        if gc != low_cls[j] or gt_matched[k]:
                            continue
                        iou = compute_iou(low_boxes[j], gb)
                        if iou > best_iou:
                            best_iou, best_gt = iou, k

                    if best_iou >= iou_threshold and best_gt >= 0:
                        xgb_tp += 1
                        gt_matched[best_gt] = True
                    else:
                        xgb_fp += 1

                xgb_fn += sum(1 for m in gt_matched if not m)
            else:
                xgb_fn += n_gt

        done = min(batch_start + batch_size, len(img_paths))
        if done % (batch_size * 10) == 0 or done >= len(img_paths):
            print(f"  Processed {done}/{len(img_paths)} images")

    # ── Compute metrics ────────────────────────────────────────────
    def metrics(tp, fp, fn):
        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        return p, r, f1

    yp, yr, yf1 = metrics(yolo_tp, yolo_fp, yolo_fn)
    xp, xr, xf1 = metrics(xgb_tp, xgb_fp, xgb_fn)

    print("\n" + "=" * 60)
    print(f"  {'Metric':<20} {'YOLO-only':>12} {'YOLO+XGBoost':>14}")
    print(f"  {'─' * 20} {'─' * 12} {'─' * 14}")
    print(f"  {'Precision':<20} {yp:>12.4f} {xp:>14.4f}")
    print(f"  {'Recall':<20} {yr:>12.4f} {xr:>14.4f}")
    print(f"  {'F1':<20} {yf1:>12.4f} {xf1:>14.4f}")
    print(f"  {'TP':<20} {yolo_tp:>12d} {xgb_tp:>14d}")
    print(f"  {'FP':<20} {yolo_fp:>12d} {xgb_fp:>14d}")
    print(f"  {'FN':<20} {yolo_fn:>12d} {xgb_fn:>14d}")
    print("=" * 60)

    return {
        'yolo': {'precision': yp, 'recall': yr, 'f1': yf1},
        'xgboost': {'precision': xp, 'recall': xr, 'f1': xf1},
    }


print("\n── Evaluating YOLO-only vs YOLO+XGBoost ──")
comparison = evaluate_pipeline(model, xgb_model, split="val", xgb_threshold=0.5)



── Evaluating YOLO-only vs YOLO+XGBoost ──

Evaluating on 2077 val images...
  Processed 320/2077 images
  Processed 640/2077 images
  Processed 960/2077 images
  Processed 1280/2077 images
  Processed 1600/2077 images
  Processed 1920/2077 images
  Processed 2077/2077 images

  Metric                  YOLO-only   YOLO+XGBoost
  ──────────────────── ──────────── ──────────────
  Precision                  0.8628         0.9474
  Recall                     0.8824         0.7989
  F1                         0.8725         0.8668
  TP                          52412          47455
  FP                           8332           2636
  FN                           6986          11943


In [ ]:
for thresh in [0.2, 0.3, 0.4, 0.5, 0.6]:
    print(f"\n── XGBoost threshold = {thresh} ──")
    evaluate_pipeline(model, xgb_model, split="val", xgb_threshold=thresh)


── XGBoost threshold = 0.2 ──

Evaluating on 2077 val images...
  Processed 320/2077 images
  Processed 640/2077 images
  Processed 960/2077 images
  Processed 1280/2077 images
  Processed 1600/2077 images
  Processed 1920/2077 images
  Processed 2077/2077 images

  Metric                  YOLO-only   YOLO+XGBoost
  ──────────────────── ──────────── ──────────────
  Precision                  0.8628         0.9012
  Recall                     0.8824         0.8604
  F1                         0.8725         0.8804
  TP                          52412          51109
  FP                           8332           5603
  FN                           6986           8289

── XGBoost threshold = 0.3 ──

Evaluating on 2077 val images...
  Processed 320/2077 images
  Processed 640/2077 images
  Processed 960/2077 images
  Processed 1280/2077 images
  Processed 1600/2077 images
  Processed 1920/2077 images
  Processed 2077/2077 images

  Metric                  YOLO-only   YOLO+XGBoost
  ───────

In [ ]:

# ══════════════════════════════════════════════════════════════════
#  STEP 7: SAVE EVERYTHING
# ══════════════════════════════════════════════════════════════════

# Save XGBoost model
xgb_model.save_model("/content/xgboost_filter.json")
print("\nXGBoost model saved → /content/xgboost_filter.json")

# Save comparison results
results = {
    "model":     "YOLOv8s + XGBoost",
    "yolo_only": {k: round(v, 4) for k, v in comparison['yolo'].items()},
    "yolo_xgb":  {k: round(v, 4) for k, v in comparison['xgboost'].items()},
}

print("\n── Final Results ──")
print(json.dumps(results, indent=2))

with open("/content/results_xgboost_pipeline.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved → /content/results_xgboost_pipeline.json")

# ── Save to Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/drive')

os.makedirs("/drive/MyDrive/manga109_xgboost", exist_ok=True)
shutil.copy("/content/xgboost_filter.json", "/drive/MyDrive/manga109_xgboost/")
shutil.copy("/content/results_xgboost_pipeline.json", "/drive/MyDrive/manga109_xgboost/")
print("Saved to Drive ✓")


XGBoost model saved → /content/xgboost_filter.json

── Final Results ──
{
  "model": "YOLOv8s + XGBoost",
  "yolo_only": {
    "precision": 0.8628,
    "recall": 0.8824,
    "f1": 0.8725
  },
  "yolo_xgb": {
    "precision": 0.9474,
    "recall": 0.7989,
    "f1": 0.8668
  }
}
Saved → /content/results_xgboost_pipeline.json
Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).
Saved to Drive ✓
